# 01 — Data Ingestion (StatsBomb Open Data)

This notebook ingests **StatsBomb Open Data** via `statsbombpy`, normalises event data, and stores outputs in:

- `data/processed/events.parquet`
- `db/football_value.duckdb` (table: `events_raw`)

> **Goal:** produce a clean, reproducible event-level table (match_id × event) that downstream notebooks can build on.


## 0. Setup

In [1]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


In [2]:
# If you haven't installed dependencies yet, run (from an activated .venv):
#   pip install -r requirements.txt

try:
    from statsbombpy import sb
except Exception as e:
    raise ImportError("statsbombpy is required. Install it with: pip install statsbombpy") from e

try:
    import duckdb
except Exception as e:
    raise ImportError("duckdb is required. Install it with: pip install duckdb") from e

from tqdm.auto import tqdm


## 1. Project paths (repo-root safe)

This avoids issues when running notebooks from different working directories (repo root vs `notebooks/`).

In [3]:
def find_repo_root(start: Optional[Path] = None) -> Path:
    """Find repo root by walking up until README.md is found."""
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
DB_DIR = REPO_ROOT / "db"

for d in [RAW_DIR, PROCESSED_DIR, DB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "football_value.duckdb"
EVENTS_PARQUET = PROCESSED_DIR / "events.parquet"

REPO_ROOT, DB_PATH, EVENTS_PARQUET


(WindowsPath('c:/Users/manue/Projects/football-possession-value'),
 WindowsPath('c:/Users/manue/Projects/football-possession-value/db/football_value.duckdb'),
 WindowsPath('c:/Users/manue/Projects/football-possession-value/data/processed/events.parquet'))

## 2. Configuration

Select competitions by **(competition_name, season_name)**. You can inspect available options in the next section.

In [4]:
@dataclass(frozen=True)
class Config:
    competitions: Tuple[Tuple[str, str], ...] = (
        ("FIFA World Cup", "2018"),
        ("La Liga", "2004/2005"),
        ("La Liga", "2005/2006"),
    )
    max_matches: Optional[int] = None  # set e.g. 10 for a quick smoke test
    save_matches_csv: bool = True

CFG = Config()
CFG


Config(competitions=(('FIFA World Cup', '2018'), ('La Liga', '2004/2005'), ('La Liga', '2005/2006')), max_matches=None, save_matches_csv=True)

## 3. Discover available competitions

In [5]:
competitions = sb.competitions()
competitions.sort_values(["competition_name", "season_name"]).head(25)


c:\Users\manue\Projects\football-possession-value\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2024-09-28T01:57:35.846538,NaN,NaN,2024-09-28T01:57:35.846538
20,16,276,Europe,Champions League,male,False,False,1970/1971,2024-02-13T14:24:12.213582,NaN,NaN,2024-02-13T14:24:12.213582
19,16,71,Europe,Champions League,male,False,False,1971/1972,2024-02-12T14:25:01.735880,2021-06-13T16:17:31.694,NaN,2024-02-12T14:25:01.735880
18,16,277,Europe,Champions League,male,False,False,1972/1973,2024-02-13T14:25:16.532771,NaN,NaN,2024-02-13T14:25:16.532771
17,16,76,Europe,Champions League,male,False,False,1999/2000,2025-06-23T12:34:36.649637,2021-06-13T16:17:31.694,NaN,2025-06-23T12:34:36.649637
16,16,44,Europe,Champions League,male,False,False,2003/2004,2025-06-24T13:57:37.321382,2021-06-13T16:17:31.694,NaN,2025-06-24T13:57:37.321382
15,16,37,Europe,Champions League,male,False,False,2004/2005,2025-06-24T09:44:09.039471,2021-06-13T16:17:31.694,NaN,2025-06-24T09:44:09.039471
14,16,39,Europe,Champions League,male,False,False,2006/2007,2024-02-12T13:48:23.967222,2021-06-13T16:17:31.694,NaN,2024-02-12T13:48:23.967222


In [6]:
wanted = set(CFG.competitions)

comp_sel = competitions[
    competitions.apply(lambda r: (r["competition_name"], r["season_name"]) in wanted, axis=1)
].copy()

if comp_sel.empty:
    raise ValueError(
        "No competitions matched CFG.competitions. "
        "Inspect sb.competitions() output and update Config."
    )

comp_sel[["competition_id", "season_id", "competition_name", "season_name"]]


,competition_id,season_id,competition_name,season_name
30,43,3,FIFA World Cup,2018
53,11,38,La Liga,2005/2006
54,11,37,La Liga,2004/2005


## 4. Download matches for selected competitions

In [7]:
all_matches = []
for _, r in comp_sel.iterrows():
    m = sb.matches(competition_id=int(r["competition_id"]), season_id=int(r["season_id"]))
    m["competition_name"] = r["competition_name"]
    m["season_name"] = r["season_name"]
    all_matches.append(m)

matches = pd.concat(all_matches, ignore_index=True)

if CFG.max_matches is not None:
    matches = matches.head(CFG.max_matches).copy()

matches.shape, matches[["match_id", "competition_name", "season_name", "home_team", "away_team", "match_date"]].head()


c:\Users\manue\Projects\football-possession-value\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\manue\Projects\football-possession-value\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\manue\Projects\football-possession-value\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


((88, 24),
    match_id competition_name season_name home_team    away_team  match_date
 0      7585   FIFA World Cup        2018  Colombia      England  2018-07-03
 1      7570   FIFA World Cup        2018   England      Belgium  2018-06-28
 2      7586   FIFA World Cup        2018    Sweden  Switzerland  2018-07-03
 3      7557   FIFA World Cup        2018      Iran     Portugal  2018-06-25
 4      7542   FIFA World Cup        2018  Portugal      Morocco  2018-06-20)

In [8]:
if CFG.save_matches_csv:
    out = PROCESSED_DIR / "matches.csv"
    matches.to_csv(out, index=False)
    print(f"Saved matches list to: {out}")


Saved matches list to: c:\Users\manue\Projects\football-possession-value\data\processed\matches.csv


## 5. Download events (match loop)

This can take a few minutes depending on the number of matches. For a quick test, set `CFG.max_matches=10`.

In [9]:
def safe_get_xy(v):
    """StatsBomb uses [x, y] lists; return (x, y) or (nan, nan)."""
    if isinstance(v, (list, tuple)) and len(v) >= 2:
        return float(v[0]), float(v[1])
    return np.nan, np.nan

events_parts = []

match_ids = matches["match_id"].astype(int).tolist()

for mid in tqdm(match_ids, desc="Downloading events"):
    ev = sb.events(match_id=mid)
    ev["match_id"] = mid
    events_parts.append(ev)

events = pd.concat(events_parts, ignore_index=True)
events.shape


c:\Users\manue\Projects\football-possession-value\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\manue\Projects\football-possession-value\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\manue\Projects\football-possession-value\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\manue\Projects\football-possession-value\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\manue\Projects\football-possession-value\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\manue\Projects\football-possession-value\.venv

(307721, 113)

## 6. Normalise columns & flatten coordinates

We keep a core schema that is stable across competitions. Downstream notebooks can add engineered features.

In [10]:
df = events.copy()

# Start coordinates
df["x_start"], df["y_start"] = zip(*df["location"].map(safe_get_xy))

# End coordinates (passes)
if "pass_end_location" in df.columns:
    px, py = zip(*df["pass_end_location"].map(safe_get_xy))
    df["x_end"], df["y_end"] = px, py
else:
    df["x_end"], df["y_end"] = np.nan, np.nan

# Carries can have their own end location
if "carry_end_location" in df.columns:
    cx, cy = zip(*df["carry_end_location"].map(safe_get_xy))
    df["x_end"] = df["x_end"].where(~df["x_end"].isna(), cx)
    df["y_end"] = df["y_end"].where(~df["y_end"].isna(), cy)

# Shots can have their own end location
if "shot_end_location" in df.columns:
    sx, sy = zip(*df["shot_end_location"].map(safe_get_xy))
    df["x_end"] = df["x_end"].where(~df["x_end"].isna(), sx)
    df["y_end"] = df["y_end"].where(~df["y_end"].isna(), sy)

core_cols = [
    "match_id",
    "id",               # event id
    "index",            # ordering index in match
    "period",
    "minute",
    "second",
    "possession",
    "possession_team",
    "team",
    "player",
    "type",
    "play_pattern",
    "under_pressure",
    "x_start", "y_start", "x_end", "y_end",
    "outcome",
]

available = [c for c in core_cols if c in df.columns]
missing = [c for c in core_cols if c not in df.columns]
print("Available core columns:", len(available))
print("Missing (ok):", missing)

core = df[available].copy()

core = core.rename(columns={"id": "event_id", "type": "event_type"})
core.head()


Available core columns: 17
Missing (ok): ['outcome']


,match_id,event_id,index,period,minute,second,possession,possession_team,team,player,event_type,play_pattern,under_pressure,x_start,y_start,x_end,y_end
0,7585,de3be98d-e227-475b-bd55-f57a6a89d308,1,1,0,0,1,Colombia,Colombia,NaN,Starting XI,Regular Play,NaN,NaN,NaN,NaN,NaN
1,7585,f50ccda4-b768-4f07-9136-8f79fd17dac5,2,1,0,0,1,Colombia,England,NaN,Starting XI,Regular Play,NaN,NaN,NaN,NaN,NaN
2,7585,b5e98805-0a22-4a5e-a306-7d40651a0f6e,3,1,0,0,1,Colombia,England,NaN,Half Start,Regular Play,NaN,NaN,NaN,NaN,NaN
3,7585,762b829f-5f24-4dd7-bfe2-da7e289838bb,4,1,0,0,1,Colombia,Colombia,NaN,Half Start,Regular Play,NaN,NaN,NaN,NaN,NaN
4,7585,6b533c4f-e8c3-4e13-94ba-791f140db2b3,1652,2,45,0,79,England,Colombia,NaN,Half Start,From Free Kick,NaN,NaN,NaN,NaN,NaN


## 7. Basic QA checks

In [11]:
print("Rows:", len(core))
print("Unique matches:", core["match_id"].nunique())
print("Event types:", core["event_type"].nunique())
core["event_type"].value_counts().head(15)


Rows: 307721
Unique matches: 88
Event types: 35


event_type
Pass              84626
Ball Receipt*     78648
Carry             67907
Pressure          29827
Ball Recovery      8537
Duel               4906
Clearance          3570
Dribble            3284
Foul Committed     2951
Block              2898
Foul Won           2815
Goal Keeper        2677
Shot               2297
Miscontrol         2273
Dribbled Past      2239
Name: count, dtype: int64

In [12]:
core = core.sort_values(["match_id", "index"], kind="mergesort").reset_index(drop=True)

dups = core.duplicated(subset=["match_id", "event_id"]).sum() if "event_id" in core.columns else None
print("Duplicate (match_id, event_id):", dups)


Duplicate (match_id, event_id): 0


## 8. Save outputs (Parquet + DuckDB)

In [13]:
core.to_parquet(EVENTS_PARQUET, index=False)
print(f"Saved: {EVENTS_PARQUET}  ({EVENTS_PARQUET.stat().st_size/1e6:.1f} MB)")


Saved: c:\Users\manue\Projects\football-possession-value\data\processed\events.parquet  (13.6 MB)


In [14]:
con = duckdb.connect(str(DB_PATH))
con.execute("CREATE OR REPLACE TABLE events_raw AS SELECT * FROM read_parquet(?)", [str(EVENTS_PARQUET)])

n = con.execute("SELECT COUNT(*) FROM events_raw").fetchone()[0]
m = con.execute("SELECT COUNT(DISTINCT match_id) FROM events_raw").fetchone()[0]
print(f"DuckDB: events_raw rows={n:,} matches={m:,}")


DuckDB: events_raw rows=307,721 matches=88


## 9. Preview from DuckDB

In [15]:
con.execute(
    """
    SELECT match_id, event_type, COUNT(*) AS n
    FROM events_raw
    GROUP BY 1,2
    ORDER BY n DESC
    LIMIT 15
    """
).df()


,match_id,event_type,n
0,7582,Pass,1483
1,7582,Ball Receipt*,1407
2,7581,Pass,1246
3,8656,Pass,1193
4,8652,Pass,1188
5,8657,Pass,1184
6,7576,Pass,1168
7,8657,Ball Receipt*,1162
8,7585,Pass,1161
9,7576,Ball Receipt*,1146


# Conclusions

## Dataset Scope

Event data was successfully ingested from **StatsBomb Open Data**, covering the following competitions:

* FIFA World Cup 2018
* La Liga 2004/2005
* La Liga 2005/2006

This selection provides a mix of **international tournament matches and domestic league games**, offering tactical diversity and sufficient action-level data for modelling possession value.

---

## Data Coverage

The ingestion pipeline produced the following dataset:

| Metric        | Value   |
| ------------- | ------- |
| Total Matches | 88      |
| Total Events  | 307,721 |
| Event Types   | 35      |

On average, the dataset contains approximately **3,500 events per match**, which is consistent with typical football event datasets.

The most frequent event types include:

* Pass
* Ball Receipt
* Carry
* Pressure
* Ball Recovery

These events are particularly relevant for modelling **ball progression and possession value**.

---

## Data Model

The ingestion pipeline generated a normalized **event-level dataset** containing key variables such as:

* match identifiers
* event sequence index
* possession identifiers
* player and team information
* spatial coordinates (`x_start`, `y_start`, `x_end`, `y_end`)
* event types and contextual attributes

This schema enables the reconstruction of:

* possession sequences
* ball progression actions
* attacking phases

which are required for possession value modelling.

---

## Data Quality Checks

Basic validation checks confirmed the integrity of the dataset:

* **No duplicated events** across `(match_id, event_id)`
* Events are **correctly ordered within matches**
* Spatial coordinates were successfully extracted from StatsBomb location arrays

These checks ensure that the dataset can be reliably used for sequential modelling.

---

## Data Persistence

The processed dataset was stored in two formats:

**Parquet**

```
data/processed/events.parquet
```

This format provides efficient storage and fast loading for pandas workflows.

**DuckDB**

```
db/football_value.duckdb
```

Table created:

```
events_raw
```

DuckDB enables fast analytical queries and will be used throughout the modelling pipeline.

---

## Output of Notebook 01

The notebook produced a **clean event-level dataset (`events_raw`) ready for analytical modelling**.

This dataset forms the foundation for the next stages of the project, including:

* possession reconstruction
* action sequence modelling
* expected threat estimation
* VAEP-style action valuation

---

## Next Step

The next notebook will focus on **event modelling and possession reconstruction**, where event sequences will be transformed into structured action-state representations required for possession value models.
